# Notebook 07 — Analytical Base Table (ABT)

**Produces two outputs:**
1. `07_analytical_base_table.csv` — ML-ready, one row per transaction enriched with archetype info
2. `07_archetype_monthly.csv` — one row per (segment × bucket × year × month) for business analysis

**Fix note:** `02_fine_bucket_ts.csv` has no Masked Range column (aggregated away in NB02).
We compute `bucket_min = floor(net_sales/qty / 100) * 100` directly on `clean_sales`.

In [1]:
import pandas as pd
import numpy as np
import pickle
import sys, os
import warnings
warnings.filterwarnings('ignore')

# Robust src path — works whether run manually or via papermill
_src_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
from config import *

OUT = OUT_PATH   # comes from config import *

clean   = pd.read_csv(f'{OUT}01_clean_sales.csv')
mapping = pd.read_csv(f'{OUT}archetype_mapping.csv')

with open(f'{OUT}03_segment_pivots.pkl', 'rb') as f:
    segment_pivots = pickle.load(f)

clean['sale_date'] = pd.to_datetime(clean['sale_date'])

print(f'clean_sales      : {clean.shape}')
print(f'archetype_mapping: {mapping.shape}')
print(f'segment_pivots   : {len(segment_pivots)} segments')
print()
print('clean_sales columns:', clean.columns.tolist())

clean_sales      : (43905, 16)
archetype_mapping: (2142, 9)
segment_pivots   : 14 segments

clean_sales columns: ['year', 'month', 'Portal', 'Brand Code Text', 'Brand(Masked)', 'Division', 'Material Number', 'Masked SKU', 'Range', 'Masked Range', 'Size', 'Final Channel', 'Final Sub Channel', 'qty', 'net_sales', 'sale_date']


## 1. Build pct_share Lookup from Segment Pivots

In [2]:
pct_rows = []
for seg_key, info in segment_pivots.items():
    pivot = info['pivot']
    div, portal, size = info['div'], info['portal'], info['size']
    for bmin in pivot.index:
        for col in pivot.columns:
            pct_rows.append({
                'Division'  : div,
                'Portal'    : portal,
                'Size'      : size,
                'bucket_min': bmin,
                'sale_date' : pd.to_datetime(col),
                'pct_share' : pivot.loc[bmin, col]
            })

pct_df = pd.DataFrame(pct_rows)
print(f'pct_share lookup rows: {len(pct_df)}')
print(pct_df.head(3).to_string(index=False))

pct_share lookup rows: 20112
Division  Portal   Size  bucket_min  sale_date  pct_share
      BP       1 Single           0 2024-01-01     0.2789
      BP       1 Single           0 2024-02-01     0.0094
      BP       1 Single           0 2024-03-01     0.2555


## 2. Compute bucket_min Directly on clean_sales

`bucket_min = floor(net_sales / qty / 100) * 100`

In [3]:
clean['ASP_raw']    = clean['net_sales'] / clean['qty']
clean['bucket_min'] = (clean['ASP_raw'] // 100) * 100

print(f'Rows with bucket_min computed : {clean["bucket_min"].notna().sum()}')
print(f'Sample bucket_min values      : {sorted(clean["bucket_min"].dropna().unique()[:10])}')

Rows with bucket_min computed : 43905
Sample bucket_min values      : [np.float64(0.0), np.float64(100.0), np.float64(400.0), np.float64(500.0), np.float64(700.0), np.float64(900.0), np.float64(1000.0), np.float64(1100.0), np.float64(1200.0), np.float64(1400.0)]


## 3. Build archetype lookup + fix outlier buckets

In [4]:
arch = mapping[['Division','Portal','Size','bucket_min',
                'New_Bucket','archetype_key']].drop_duplicates()

# Highest bucket per segment — used to fix outliers below
seg_max_bucket = (arch.groupby(['Division','Portal','Size'])['bucket_min']
                  .max().reset_index()
                  .rename(columns={'bucket_min':'max_bucket_min'}))

def fix_outliers(df, arch, seg_max_bucket):
    """Assign extreme outlier prices (beyond mapping range) to segment's highest bucket."""
    unmatched = df['archetype_key'].isna()
    if not unmatched.any():
        return df
    fix = df[unmatched].copy().drop(columns=['New_Bucket','archetype_key'])
    fix = fix.merge(seg_max_bucket, on=['Division','Portal','Size'], how='left')
    fix['bucket_min'] = fix['max_bucket_min']
    fix = fix.drop(columns=['max_bucket_min'])
    fix = fix.merge(arch, on=['Division','Portal','Size','bucket_min'], how='left')
    return pd.concat([df[~unmatched], fix], ignore_index=True)

print('arch lookup rows:', len(arch))

arch lookup rows: 2142


## 4. Enrich clean_sales with archetype_key + pct_share

In [5]:
abt = clean.merge(arch, on=['Division','Portal','Size','bucket_min'], how='left')
abt = fix_outliers(abt, arch, seg_max_bucket)

abt = abt.merge(pct_df, on=['Division','Portal','Size','bucket_min','sale_date'], how='left')

print(f'Rows total             : {len(abt)}')
print(f'Missing archetype_key  : {abt["archetype_key"].isna().sum()}')
print(f'Missing pct_share      : {abt["pct_share"].isna().sum()} ({abt["pct_share"].isna().mean():.2%})')

Rows total             : 43905
Missing archetype_key  : 0
Missing pct_share      : 1188 (2.71%)


## 5. Add Segment-Level Monthly Aggregates + Derived Features

In [6]:
seg_monthly = (abt.groupby(['Division','Portal','Size','sale_date'])
               .agg(segment_total_qty   =('qty',      'sum'),
                    segment_total_sales =('net_sales', 'sum'))
               .reset_index())
abt = abt.merge(seg_monthly, on=['Division','Portal','Size','sale_date'], how='left')

abt['year']          = abt['sale_date'].dt.year
abt['month']         = abt['sale_date'].dt.month
abt['ASP']           = np.where(abt['qty'] > 0, abt['net_sales'] / abt['qty'], np.nan)
abt['bucket_max']    = abt['bucket_min'] + (BUCKET_WIDTH - 1)
abt['row_qty_share'] = np.where(abt['segment_total_qty'] > 0,
                                abt['qty'] / abt['segment_total_qty'], np.nan)

print(f'ABT shape: {abt.shape}')

ABT shape: (43905, 26)


## 6. Final Column Order, Quality Check & Save ABT

In [7]:
col_order = ['Division','Portal','Size','sale_date','year','month',
             RANGE_COL,'bucket_min','bucket_max','ASP',
             'New_Bucket','archetype_key',
             'pct_share','row_qty_share',
             'qty','net_sales','segment_total_qty','segment_total_sales']
col_order = [c for c in col_order if c in abt.columns]
abt_final = abt[col_order].copy()

print('=== ABT QUALITY SUMMARY ===')
print(f'Shape                  : {abt_final.shape}')
print(f'Date range             : {abt_final["sale_date"].min()} \u2192 {abt_final["sale_date"].max()}')
print(f'Segments               : {abt_final.groupby(["Division","Portal","Size"]).ngroups}')
print(f'Unique archetype keys  : {abt_final["archetype_key"].nunique()}')
print(f'Rows with archetype    : {abt_final["archetype_key"].notna().mean():.1%}')
print(f'Total qty              : {abt_final["qty"].sum():,.0f}')
print(f'Total net_sales        : \u20b9{abt_final["net_sales"].sum():,.0f}')
print()
nulls = abt_final.isnull().sum()
print('Nulls:', nulls[nulls > 0].to_string() if nulls.any() else 'None \u2713')
print()
out_path = f'{OUT}07_analytical_base_table.csv'
abt_final.to_csv(out_path, index=False)
print(f'\u2713 Saved \u2192 {out_path}  ({abt_final.shape[0]:,} rows \u00d7 {abt_final.shape[1]} cols)')
print(f'  File size: ~{os.path.getsize(out_path) / 1e6:.1f} MB')

=== ABT QUALITY SUMMARY ===
Shape                  : (43905, 18)


Date range             : 2024-01-01 00:00:00 → 2025-12-01 00:00:00
Segments               : 14
Unique archetype keys  : 96
Rows with archetype    : 100.0%
Total qty              : 4,620,619
Total net_sales        : ₹9,670,635,150

Nulls: pct_share    1188



✓ Saved → data/outputs/TT/07_analytical_base_table.csv  (43,905 rows × 18 cols)
  File size: ~6.2 MB


---
## 7. Monthly Archetype Breakdown

**Goal:** Automated equivalent of the analyst's manual Excel pivot.
One row per (Division × Portal × Size × bucket_min × year × month).

Columns: `Division, Portal, Size, year, month, bucket_min, New_Bucket, archetype_key, qty, net_sales, ASP`

**Use in Excel:** PivotTable → Rows: `archetype_key`, Columns: `year + month`, Values: `qty`

In [8]:
# Aggregate clean_sales to segment × bucket × year × month
monthly = (clean.groupby(['Division','Portal','Size','bucket_min','year','month'])
           .agg(qty      =('qty',      'sum'),
                net_sales=('net_sales', 'sum'))
           .reset_index())

# Join archetype info + fix outliers
monthly = monthly.merge(arch, on=['Division','Portal','Size','bucket_min'], how='left')
monthly = fix_outliers(monthly, arch, seg_max_bucket)

monthly['ASP']        = (monthly['net_sales'] / monthly['qty']).round(2)
monthly['New_Bucket'] = monthly['New_Bucket'].astype(int)

monthly_final = (monthly[['Division','Portal','Size','year','month',
                           'bucket_min','New_Bucket','archetype_key',
                           'qty','net_sales','ASP']]
                 .sort_values(['Division','Portal','Size','bucket_min','year','month'])
                 .reset_index(drop=True))

print('=== MONTHLY BREAKDOWN SUMMARY ===')
print(f'Shape                : {monthly_final.shape}')
print(f'Segments             : {monthly_final.groupby(["Division","Portal","Size"]).ngroups}')
print(f'Unique archetype keys: {monthly_final["archetype_key"].nunique()}')
print(f'Year/month combos    : {monthly_final.groupby(["year","month"]).ngroups} (expect 24)')
print()
print('=== PREVIEW — HL / CABIN ===')
preview = monthly_final[
    (monthly_final['Division']=='HL') &
    (monthly_final['Size']=='CABIN') &
    (monthly_final['bucket_min']==1200)
]
print(preview.to_string(index=False))

=== MONTHLY BREAKDOWN SUMMARY ===
Shape                : (10011, 11)
Segments             : 14
Unique archetype keys: 96
Year/month combos    : 24 (expect 24)

=== PREVIEW — HL / CABIN ===
Division  Portal  Size  year  month  bucket_min  New_Bucket archetype_key   qty   net_sales     ASP
      HL       1 CABIN  2024      1      1200.0           2   HLTT12CABIN  1416  1812931.96 1280.32
      HL       1 CABIN  2024      4      1200.0           2   HLTT12CABIN   413   509827.84 1234.45
      HL       1 CABIN  2024      5      1200.0           2   HLTT12CABIN   376   472884.25 1257.67
      HL       1 CABIN  2024      6      1200.0           2   HLTT12CABIN   259   322946.28 1246.90
      HL       1 CABIN  2024      7      1200.0           2   HLTT12CABIN   618   763300.21 1235.11
      HL       1 CABIN  2024      8      1200.0           2   HLTT12CABIN   288   356296.42 1237.14
      HL       1 CABIN  2024      9      1200.0           2   HLTT12CABIN   484   596743.13 1232.94
      HL   

## 8. Save Monthly Breakdown

In [9]:
monthly_path = f'{OUT}07_archetype_monthly.csv'
monthly_final.to_csv(monthly_path, index=False)
print(f'\u2713 Saved \u2192 {monthly_path}')
print(f'  {monthly_final.shape[0]:,} rows \u00d7 {monthly_final.shape[1]} columns')
print()
print('HOW TO PIVOT IN EXCEL:')
print('  Open 07_archetype_monthly.csv \u2192 Insert \u2192 PivotTable')
print('  Rows    : archetype_key  (or New_Bucket)')
print('  Columns : year + month')
print('  Values  : Sum of qty  (or net_sales / ASP)')
print('  Filter  : Division, Size to drill into one segment')
print()
print('\u2713 NB07 complete. Outputs saved:')
print(f'  {OUT}07_analytical_base_table.csv')
print(f'  {OUT}07_archetype_monthly.csv')

✓ Saved → data/outputs/TT/07_archetype_monthly.csv
  10,011 rows × 11 columns

HOW TO PIVOT IN EXCEL:
  Open 07_archetype_monthly.csv → Insert → PivotTable
  Rows    : archetype_key  (or New_Bucket)
  Columns : year + month
  Values  : Sum of qty  (or net_sales / ASP)
  Filter  : Division, Size to drill into one segment

✓ NB07 complete. Outputs saved:
  data/outputs/TT/07_analytical_base_table.csv
  data/outputs/TT/07_archetype_monthly.csv
